## setp


In [ ]:
import os
import jax
import flax
import tyro
import time
import optax
import wandb
import pickle
import random
import wandb_osh
import numpy as np
import flax.linen as nn
import jax.numpy as jnp
import distrax

from brax import envs
from etils import epath
from dataclasses import dataclass
from collections import namedtuple
from typing import NamedTuple, Any
from wandb_osh.hooks import TriggerWandbSyncHook
from flax.training.train_state import TrainState
from flax.linen.initializers import variance_scaling

from evaluator import CrlEvaluator
from buffer import TrajectoryUniformSamplingQueue

import matplotlib.pyplot as plt

#with the multi-policy idea, all that would be need to be changed would be many different actor_params in the actor_state

#list of modifications so far:
#actor_step, deterministic_actor_step, simple env if statement, replay buffer initialization, checkpoint loading

#intended modifications:
#decrease the max replay size in accordance with num agents
#notice that this gives the same effect as not making new buffers for agents and
#instead putting the agent trajectories one after the other in the same buffer

@dataclass
class Args:
    exp_name: str = os.path.basename(__file__)[: -len(".py")]
    seed: int = 1
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = False
    wandb_project_name: str = "TEST_WANDB"
    wandb_entity: str = 'asim_awad'
    wandb_mode: str = 'offline'
    wandb_dir: str = '.'
    wandb_group: str = '.'
    capture_video: bool = False
    checkpoint: bool = False
    load_path: str = ''

    #environment specific arguments
    env_id: str = "smax"
    episode_length: int = 101
    # to be filled in runtime
    obs_dim: int = 0
    goal_start_idx: int = 0
    goal_end_idx: int = 0
    num_agents: int = 0
    discrete_actions: bool = False

    # Algorithm specific arguments
    total_env_steps: int = 50_000_000
    num_epochs: int = 500
    num_envs: int = 256
    num_eval_envs: int = 64
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    batch_size: int = 256
    gamma: float = 0.99
    logsumexp_penalty_coeff: float = 0.1

    max_replay_size: int = 5000 #temporarily reduced by 2 to accomodate more agents, should make this systematic in future
    min_replay_size: int = 1000
    
    unroll_length: int  = 62

    # to be filled in runtime
    env_steps_per_actor_step : int = 0
    """number of env steps per actor step (computed in runtime)"""
    num_prefill_env_steps : int = 0
    """number of env steps to fill the buffer before starting training (computed in runtime)"""
    num_prefill_actor_steps : int = 0
    """number of actor steps to fill the buffer before starting training (computed in runtime)"""
    num_training_steps_per_epoch : int = 0
    """the number of training steps per epoch(computed in runtime)"""
    num_envs_agents : int = 0
    """the number of agents times the number of environments (2nd dimension replay buffer)"""

class SA_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = jnp.concatenate([s, a], axis=-1)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x
    
class G_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, g: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(g)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x

class Actor(nn.Module):
    action_size: int
    norm_type = "layer_norm"

    LOG_STD_MAX = 2
    LOG_STD_MIN = -5

    @nn.compact
    def __call__(self, x):
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)

        mean = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        log_std = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        
        log_std = nn.tanh(log_std)
        log_std = self.LOG_STD_MIN + 0.5 * (self.LOG_STD_MAX - self.LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

@flax.struct.dataclass
class TrainingState:
    """Contains training state for the learner"""
    env_steps: jnp.ndarray
    gradient_steps: jnp.ndarray
    actor_state: TrainState
    critic_state: TrainState
    alpha_state: TrainState

class Transition(NamedTuple):
    """Container for a transition"""
    observation: jnp.ndarray
    action: jnp.ndarray
    reward: jnp.ndarray
    discount: jnp.ndarray
    extras: jnp.ndarray = ()  

def load_params(path: str):
    with epath.Path(path).open('rb') as fin:
        buf = fin.read()
    return pickle.loads(buf)

def save_params(path: str, params: Any):
    """Saves parameters in flax format."""
    with epath.Path(path).open('wb') as fout:
        fout.write(pickle.dumps(params))

In [23]:

args = Args()

args.env_steps_per_actor_step = args.num_envs * args.unroll_length
args.num_prefill_env_steps = args.min_replay_size * args.num_envs
args.num_prefill_actor_steps = np.ceil(args.min_replay_size / args.unroll_length)
args.num_training_steps_per_epoch = (args.total_env_steps - args.num_prefill_env_steps) // (args.num_epochs * args.env_steps_per_actor_step)

run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

if args.track:

    if args.wandb_group ==  '.':
        args.wandb_group = None
        
    wandb.init(
        project=args.wandb_project_name,
        entity=args.wandb_entity,
        mode=args.wandb_mode,
        group=args.wandb_group,
        dir=args.wandb_dir,
        config=vars(args),
        name=run_name,
        monitor_gym=True,
        save_code=True,
    )

    if args.wandb_mode == 'offline':
        wandb_osh.set_log_level("ERROR")
        trigger_sync = TriggerWandbSyncHook()
    
if args.checkpoint:
    from pathlib import Path
    save_path = Path(args.wandb_dir) / Path(run_name)
    os.mkdir(path=save_path)

random.seed(args.seed)
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, buffer_key, env_key, eval_env_key, actor_key, sa_key, g_key = jax.random.split(key, 7)

# Environment setup    
if args.env_id == "ant":
    from envs.ant import Ant
    env = Ant(
        backend="spring",
        exclude_current_positions_from_observation=False,
        terminate_when_unhealthy=True,
    )

    args.obs_dim = 29
    args.goal_start_idx = 0
    args.goal_end_idx = 2

elif "maze" in args.env_id:
    from envs.ant_maze import AntMaze
    env = AntMaze(
        backend="spring",
        exclude_current_positions_from_observation=False,
        terminate_when_unhealthy=True,
        maze_layout_name=args.env_id[4:]
    )

    args.obs_dim = 29
    args.goal_start_idx = 0
    args.goal_end_idx = 2

elif args.env_id == "simple_marl":
    from envs.mpe_simple_marl import SimpleMPEMARL
    env = SimpleMPEMARL()
    args.obs_dim = 4
    args.goal_start_idx = 2
    args.goal_end_idx = 4
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "push_marl":
    from envs.mpe_push_LLM import PushMPEMARL
    env = PushMPEMARL()
    args.obs_dim = 6  # vel(2) + landmark_pos(4)
    args.goal_start_idx = 6  # target position starts after obs
    args.goal_end_idx = 8   # target is 2D position
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mpe_tag":
    from envs.mpe_tag import MPETagCoop
    env = MPETagCoop()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_adversaries
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mpe_tag_facmac":
    from envs.mpe_tag_facmac import MPETagFacmac
    env = MPETagFacmac()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_adversaries
    args.num_envs_agents = args.num_envs * args.num_agents
elif args.env_id == "mpe_tag_facmac_6a":
    from envs.mpe_tag_facmac_6a import MPETagFacmac6a
    env = MPETagFacmac6a()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_adversaries
    args.num_envs_agents = args.num_envs * args.num_agents
elif args.env_id == "smax":
    from envs.smax import SmaxEnv
    env = SmaxEnv()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents
    args.discrete_actions = False #for now

elif args.env_id == "smax_move":
    from envs.smax_move import SmaxEnv
    env = SmaxEnv()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents
    args.discrete_actions = False #for now

elif args.env_id == "mabrax_ant":
    from envs.mabrax_ant import MABraxAnt
    env = MABraxAnt()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mabrax_walker":
    from envs.mabrax_walker import MABraxWalker
    env = MABraxWalker()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

else:
    raise NotImplementedError

#new modification for unavail action penalization
if ('smax' in args.env_id) and args.discrete_actions:
    get_avail_act = jax.vmap(env.get_avail_actions)

env = envs.training.wrap(
    env,
    episode_length=args.episode_length,
)

obs_size = env.observation_size
action_size = env.action_size
env_keys = jax.random.split(env_key, args.num_envs)
env_state = jax.jit(env.reset)(env_keys)
env.step = jax.jit(env.step)

#load checkpoint to resume training
alph_params, act_params, crtc_params = load_params(args.load_path) if args.load_path else [None]*3

# Network setup
# Actor
actor = Actor(action_size=action_size)
if not act_params:
    act_params = actor.init(actor_key, np.ones([1, obs_size]))
actor_state = TrainState.create(
    apply_fn=actor.apply,
    params=act_params,
    tx=optax.adam(learning_rate=args.actor_lr)
)

# Critic
sa_encoder = SA_encoder()
sa_encoder_params = sa_encoder.init(sa_key, np.ones([1, args.obs_dim]), np.ones([1, action_size]))
g_encoder = G_encoder()
g_encoder_params = g_encoder.init(g_key, np.ones([1, args.goal_end_idx - args.goal_start_idx]))
c = jnp.asarray(0.0, dtype=jnp.float32)
if not crtc_params:
    crtc_params = {"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params}
critic_state = TrainState.create(
    apply_fn=None,
    params={"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params},
    tx=optax.adam(learning_rate=args.critic_lr),
)

# Entropy coefficient
target_entropy = -0.5 * action_size
log_alpha = jnp.asarray(0.0, dtype=jnp.float32)
if not alph_params:
    alph_params = {"log_alpha": log_alpha}
alpha_state = TrainState.create(
    apply_fn=None,
    params=alph_params,
    tx=optax.adam(learning_rate=args.alpha_lr),
)

# Trainstate
training_state = TrainingState(
    env_steps=jnp.zeros(()),
    gradient_steps=jnp.zeros(()),
    actor_state=actor_state,
    critic_state=critic_state,
    alpha_state=alpha_state,
)

#Replay Buffer
dummy_obs = jnp.zeros((obs_size,))
dummy_action = jnp.zeros((action_size,))

dummy_transition = Transition(
    observation=dummy_obs,
    action=dummy_action,
    reward=0.0,
    discount=0.0,
    extras={
        "state_extras": {
            "truncation": 0.0,
            "seed": 0.0,
        }        
    },
)

def jit_wrap(buffer):
    buffer.insert_internal = jax.jit(buffer.insert_internal)
    buffer.sample_internal = jax.jit(buffer.sample_internal)
    return buffer

#change replay buffer to make the 2nd dimension num_envs_agents
replay_buffer = jit_wrap(
        TrajectoryUniformSamplingQueue(
            max_replay_size=args.max_replay_size,
            dummy_data_sample=dummy_transition,
            sample_batch_size=args.batch_size,
            num_envs=args.num_envs_agents,
            episode_length=args.episode_length,
        )
    )
buffer_state = jax.jit(replay_buffer.init)(buffer_key)

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def deterministic_actor_step(training_state, env, env_state, extra_fields):
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, _ = actor.apply(training_state.actor_state.params, obs)
    action = None
    
    if not args.discrete_actions:
        actions = nn.tanh( means )
    else:
        actions = jnp.argmax(means, axis=-1)
    
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
    nstate = env.step(env_state, actions_)
    
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}
    return nstate, Transition(
        observation=obs,
        action=actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def actor_step(actor_state, env, env_state, key, extra_fields):
    #perform inference to get actions
    keys = jax.random.split(key, args.num_agents)
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, log_stds = actor.apply(actor_state.params, obs)
    actions = None
    transition_actions = None
    
    if not args.discrete_actions:
        stds = jnp.exp(log_stds)
        noise = jax.vmap(lambda rng: jax.random.normal(
                    rng, shape=(args.num_envs,) + means.shape[1:], 
                    dtype=means.dtype), in_axes=0, out_axes=1)(keys)
        noise_ = jnp.reshape(noise, (-1,) + noise.shape[2:])
        actions = nn.tanh( means + stds * noise_)
        transition_actions = actions
    else:
        avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
        avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
        action_logits = means - ((1-avail_actions)*1e10)
        
        pi = distrax.Categorical(logits = action_logits)
        actions = pi.sample(seed = keys[0])
        transition_actions = jax.nn.one_hot(actions, means.shape[1])
        
    #step our environment
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
#        jax.debug.print("actions from first env: {}", transition_actions[0])
    nstate = env.step(env_state, actions_)
    
    #generate an array of transitions, with shape (num_envs*num_agents,...)
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}

    return nstate, Transition(
        observation=obs,
        action=transition_actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

@jax.jit
def get_experience(actor_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused_t):
        env_state, current_key = carry
        current_key, next_key = jax.random.split(current_key)
        env_state, transition = actor_step(actor_state, env, env_state, current_key, extra_fields=("truncation", "seed"))
        return (env_state, next_key), transition

    (env_state, _), data = jax.lax.scan(f, (env_state, key), (), length=args.unroll_length)
    buffer_state = replay_buffer.insert(buffer_state, data)
    return env_state, buffer_state

def prefill_replay_buffer(training_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused):
        del unused
        training_state, env_state, buffer_state, key = carry
        key, new_key = jax.random.split(key)
        env_state, buffer_state = get_experience(
            training_state.actor_state,
            env_state,
            buffer_state,
            key,
        
        )
        training_state = training_state.replace(
            env_steps=training_state.env_steps + args.env_steps_per_actor_step,
        )
        return (training_state, env_state, buffer_state, new_key), ()

    return jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_prefill_actor_steps)[0]

#modified to accomodate discrete action space
@jax.jit
def update_actor_and_alpha(transitions, training_state, key):
    def actor_loss(actor_params, critic_params, log_alpha, transitions, key):
        obs = transitions.observation           # expected_shape = batch_size, obs_size + goal_size
        state = obs[:, :args.obs_dim]
        future_state = transitions.extras["future_state"]
        goal = future_state[:, args.goal_start_idx : args.goal_end_idx]
        observation = jnp.concatenate([state, goal], axis=1)
        jax.debug.print("observation: {}", observation.shape)
        jax.debug.print("observation: {}", observation[0,-2:])

        means, log_stds = actor.apply(actor_params, observation)
        action = None
        log_prob = None
        
        if not args.discrete_actions:
            stds = jnp.exp(log_stds)
            x_ts = means + stds * jax.random.normal(key, shape=means.shape, dtype=means.dtype)
            action = nn.tanh(x_ts)
            log_prob = jax.scipy.stats.norm.logpdf(x_ts, loc=means, scale=stds)
            log_prob -= jnp.log((1 - jnp.square(action)) + 1e-6)
            log_prob = log_prob.sum(-1)           # dimension = B
        else:
            pi = distrax.Categorical(logits = means)
            action = pi.sample(seed = key)
#                jax.debug.print("actions on sampled obs: {}", means[:6,:])
            log_prob = pi.log_prob(action)
            action = jax.nn.one_hot(action, means.shape[1])

        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        sa_repr = sa_encoder.apply(sa_encoder_params, state, action)
        g_repr = g_encoder.apply(g_encoder_params, goal)

        qf_pi = -jnp.sqrt(jnp.sum((sa_repr - g_repr) ** 2, axis=-1))

        actor_loss = jnp.mean( jnp.exp(log_alpha) * log_prob - (qf_pi) )

        return actor_loss, log_prob

    def alpha_loss(alpha_params, log_prob):
        alpha = jnp.exp(alpha_params["log_alpha"])
        alpha_loss = alpha * jnp.mean(jax.lax.stop_gradient(-log_prob - target_entropy))
        return jnp.mean(alpha_loss)
    
    (actorloss, log_prob), actor_grad = jax.value_and_grad(actor_loss, has_aux=True)(training_state.actor_state.params, training_state.critic_state.params, training_state.alpha_state.params['log_alpha'], transitions, key)
    new_actor_state = training_state.actor_state.apply_gradients(grads=actor_grad)

    alphaloss, alpha_grad = jax.value_and_grad(alpha_loss)(training_state.alpha_state.params, log_prob)
    new_alpha_state = training_state.alpha_state.apply_gradients(grads=alpha_grad)

    training_state = training_state.replace(actor_state=new_actor_state, alpha_state=new_alpha_state)

    metrics = {
        "sample_entropy": -log_prob,
        "actor_loss": actorloss,
        "alph_aloss": alphaloss,   
        "log_alpha": training_state.alpha_state.params["log_alpha"],
    }

    return training_state, metrics

@jax.jit
def update_critic(transitions, training_state, key):
    def critic_loss(critic_params, transitions, key):
        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        
        obs = transitions.observation[:, :args.obs_dim]
        action = transitions.action
        
        sa_repr = sa_encoder.apply(sa_encoder_params, obs, action)
        g_repr = g_encoder.apply(g_encoder_params, transitions.observation[:, args.obs_dim:])
        
        # InfoNCE
        logits = -jnp.sqrt(jnp.sum((sa_repr[:, None, :] - g_repr[None, :, :]) ** 2, axis=-1))       # shape = BxB
        critic_loss = -jnp.mean(jnp.diag(logits) - jax.nn.logsumexp(logits, axis=1))

        # logsumexp regularisation
        logsumexp = jax.nn.logsumexp(logits + 1e-6, axis=1)
        critic_loss += args.logsumexp_penalty_coeff * jnp.mean(logsumexp**2)

        I = jnp.eye(logits.shape[0])
        correct = jnp.argmax(logits, axis=1) == jnp.argmax(I, axis=1)
        logits_pos = jnp.sum(logits * I) / jnp.sum(I)
        logits_neg = jnp.sum(logits * (1 - I)) / jnp.sum(1 - I)

        return critic_loss, (logsumexp, I, correct, logits_pos, logits_neg)
        
    (loss, (logsumexp, I, correct, logits_pos, logits_neg)), grad = jax.value_and_grad(critic_loss, has_aux=True)(training_state.critic_state.params, transitions, key)
    new_critic_state = training_state.critic_state.apply_gradients(grads=grad)
    training_state = training_state.replace(critic_state = new_critic_state)

    metrics = {
        "categorical_accuracy": jnp.mean(correct),
        "logits_pos": logits_pos,
        "logits_neg": logits_neg,
        "logsumexp": logsumexp.mean(),
        "critic_loss": loss,
    }

    return training_state, metrics

@jax.jit
def sgd_step(carry, transitions):
    training_state, key = carry
    key, critic_key, actor_key, = jax.random.split(key, 3)

    training_state, actor_metrics = update_actor_and_alpha(transitions, training_state, actor_key)

    training_state, critic_metrics = update_critic(transitions, training_state, critic_key)

    training_state = training_state.replace(gradient_steps = training_state.gradient_steps + 1)

    metrics = {}
    metrics.update(actor_metrics)
    metrics.update(critic_metrics)
    
    return (training_state, key,), metrics

@jax.jit
def training_step(training_state, env_state, buffer_state, key):
    experience_key1, experience_key2, sampling_key, training_key = jax.random.split(key, 4)

    # update buffer
    env_state, buffer_state = get_experience(
        training_state.actor_state,
        env_state,
        buffer_state,
        experience_key1,
    )

    training_state = training_state.replace(
        env_steps=training_state.env_steps + args.env_steps_per_actor_step,
    )

    # sample actor-step worth of transitions
    buffer_state, transitions = replay_buffer.sample(buffer_state)

    # process transitions for training
    batch_keys = jax.random.split(sampling_key, transitions.observation.shape[0])
    transitions = jax.vmap(TrajectoryUniformSamplingQueue.flatten_crl_fn, in_axes=(None, 0, 0))(
        (args.gamma, args.obs_dim, args.goal_start_idx, args.goal_end_idx), transitions, batch_keys
    )
    
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1,) + x.shape[2:], order="F"),
        transitions,
    )
    permutation = jax.random.permutation(experience_key2, len(transitions.observation))
    transitions = jax.tree_util.tree_map(lambda x: x[permutation], transitions)
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1, args.batch_size) + x.shape[1:]),
        transitions,
    )

    # take actor-step worth of training-step
    (training_state, _,), metrics = jax.lax.scan(sgd_step, (training_state, training_key), transitions)

    return (training_state, env_state, buffer_state,), metrics

@jax.jit
def training_epoch(
    training_state,
    env_state,
    buffer_state,
    key,
):  
    @jax.jit
    def f(carry, unused_t):
        ts, es, bs, k = carry
        k, train_key = jax.random.split(k, 2)
        (ts, es, bs,), metrics = training_step(ts, es, bs, train_key)
        return (ts, es, bs, k), metrics

    (training_state, env_state, buffer_state, key), metrics = jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_training_steps_per_epoch)
    
    metrics["buffer_current_size"] = replay_buffer.size(buffer_state)
    return training_state, env_state, buffer_state, metrics

key, prefill_key = jax.random.split(key, 2)

training_state, env_state, buffer_state, _ = prefill_replay_buffer(
    training_state, env_state, buffer_state, prefill_key
)

'''Setting up evaluator'''
evaluator = CrlEvaluator(
    deterministic_actor_step,
    env,
    num_eval_envs=args.num_eval_envs,
    episode_length=args.episode_length,
    key=eval_env_key,
)


In [ ]:

training_walltime = 0
print('starting training....')
for ne in range(args.num_epochs):
    t = time.time()

    key, epoch_key = jax.random.split(key)
    training_state, env_state, buffer_state, metrics = training_epoch(training_state, env_state, buffer_state, epoch_key)
    
    metrics = jax.tree_util.tree_map(jnp.mean, metrics)
    metrics = jax.tree_util.tree_map(lambda x: x.block_until_ready(), metrics)

    epoch_training_time = time.time() - t
    training_walltime += epoch_training_time

    sps = (args.env_steps_per_actor_step * args.num_training_steps_per_epoch) / epoch_training_time
    metrics = {
        "training/sps": sps,
        "training/walltime": training_walltime,
        "training/envsteps": training_state.env_steps.item(),
        **{f"training/{name}": value for name, value in metrics.items()},
    }

    # update metrics, get EVAL episode returns
    metrics = evaluator.run_evaluation(training_state, metrics)

starting training....


TypeError: cannot reshape array of shape (1000, 129) (size 129000) into shape (-1, 256, 129) because the product of specified axis sizes (33024) does not evenly divide 129000

: 

In [6]:
import os
import jax
import flax
import tyro
import time
import optax
import wandb
import pickle
import random
import wandb_osh
import numpy as np
import flax.linen as nn
import jax.numpy as jnp
import distrax

from brax import envs
from etils import epath
from dataclasses import dataclass
from collections import namedtuple
from typing import NamedTuple, Any
from wandb_osh.hooks import TriggerWandbSyncHook
from flax.training.train_state import TrainState
from flax.linen.initializers import variance_scaling

from evaluator import CrlEvaluator
from buffer import TrajectoryUniformSamplingQueue

import matplotlib.pyplot as plt




@dataclass
class Args:
    exp_name: str = "testing"
    seed: int = 1
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = True
    wandb_project_name: str = "ICRL_Reproduction"
    wandb_entity: str = 'asim_awad'
    wandb_mode: str = 'offline'
    wandb_dir: str = '.'
    wandb_group: str = '.'
    capture_video: bool = False
    checkpoint: bool = False
    load_path: str = ''

    #environment specific arguments
    env_id: str = "smax"
    episode_length: int = 101
    # to be filled in runtime
    obs_dim: int = 0
    goal_start_idx: int = 0
    goal_end_idx: int = 0
    num_agents: int = 0
    discrete_actions: bool = False

    # Algorithm specific arguments
    total_env_steps: int = 50_000_000
    num_epochs: int = 500
    num_envs: int = 2 #256
    num_eval_envs: int = 64
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    batch_size: int = 256
    gamma: float = 0.99
    logsumexp_penalty_coeff: float = 0.1

    max_replay_size: int = 5000 
    min_replay_size: int = 1000
    
    unroll_length: int  = 62

    # to be filled in runtime
    env_steps_per_actor_step : int = 0
    """number of env steps per actor step (computed in runtime)"""
    num_prefill_env_steps : int = 0
    """number of env steps to fill the buffer before starting training (computed in runtime)"""
    num_prefill_actor_steps : int = 0
    """number of actor steps to fill the buffer before starting training (computed in runtime)"""
    num_training_steps_per_epoch : int = 0
    """the number of training steps per epoch(computed in runtime)"""
    num_envs_agents : int = 0
    """the number of agents times the number of environments (2nd dimension replay buffer)"""




In [7]:
class SA_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = jnp.concatenate([s, a], axis=-1)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x
    
class G_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, g: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(g)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x


@flax.struct.dataclass
class TrainingState:
    """Contains training state for the learner"""
    env_steps: jnp.ndarray
    gradient_steps: jnp.ndarray
    actor_state: TrainState
    critic_state: TrainState
    alpha_state: TrainState

class Transition(NamedTuple):
    """Container for a transition"""
    observation: jnp.ndarray
    action: jnp.ndarray
    reward: jnp.ndarray
    discount: jnp.ndarray
    extras: jnp.ndarray = ()  

def load_params(path: str):
    with epath.Path(path).open('rb') as fin:
        buf = fin.read()
    return pickle.loads(buf)

def save_params(path: str, params: Any):
    """Saves parameters in flax format."""
    with epath.Path(path).open('wb') as fout:
        fout.write(pickle.dumps(params))
                   


In [8]:
args = Args()

args.env_steps_per_actor_step = args.num_envs * args.unroll_length
args.num_prefill_env_steps = args.min_replay_size * args.num_envs
args.num_prefill_actor_steps = np.ceil(args.min_replay_size / args.unroll_length)
args.num_training_steps_per_epoch = (args.total_env_steps - args.num_prefill_env_steps) // (args.num_epochs * args.env_steps_per_actor_step)

run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

for key, value in vars(args).items():
    print(f"{key}: {value}")

exp_name: testing
seed: 1
torch_deterministic: True
cuda: True
track: True
wandb_project_name: ICRL_Reproduction
wandb_entity: asim_awad
wandb_mode: offline
wandb_dir: .
wandb_group: .
capture_video: False
checkpoint: False
load_path: 
env_id: smax
episode_length: 101
obs_dim: 0
goal_start_idx: 0
goal_end_idx: 0
num_agents: 0
discrete_actions: False
total_env_steps: 50000000
num_epochs: 500
num_envs: 2
num_eval_envs: 64
actor_lr: 0.0003
critic_lr: 0.0003
alpha_lr: 0.0003
batch_size: 256
gamma: 0.99
logsumexp_penalty_coeff: 0.1
max_replay_size: 5000
min_replay_size: 1000
unroll_length: 62
env_steps_per_actor_step: 124
num_prefill_env_steps: 2000
num_prefill_actor_steps: 17.0
num_training_steps_per_epoch: 806
num_envs_agents: 0


In [9]:

if args.track:

    if args.wandb_group ==  '.':
        args.wandb_group = None
        
    wandb.init(
        project=args.wandb_project_name,
        entity=args.wandb_entity,
        mode=args.wandb_mode,
        group=args.wandb_group,
        dir=args.wandb_dir,
        config=vars(args),
        name=run_name,
        monitor_gym=True,
        save_code=True,
    )

    if args.wandb_mode == 'offline':
        wandb_osh.set_log_level("ERROR")
        trigger_sync = TriggerWandbSyncHook()
    
if args.checkpoint:
    from pathlib import Path
    save_path = Path(args.wandb_dir) / Path(run_name)
    os.mkdir(path=save_path)

random.seed(args.seed)
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, buffer_key, env_key, eval_env_key, actor_key, sa_key, g_key = jax.random.split(key, 7)

if args.env_id == "smax":
    from envs.smax import SmaxEnv
    env = SmaxEnv()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents
    args.discrete_actions = False #for now

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


In [10]:
print("DERIVED:", {
  "env_steps_per_actor_step": args.env_steps_per_actor_step,
  "num_prefill_env_steps": args.num_prefill_env_steps,
  "num_prefill_actor_steps": args.num_prefill_actor_steps,
  "num_training_steps_per_epoch": args.num_training_steps_per_epoch,
})


DERIVED: {'env_steps_per_actor_step': 124, 'num_prefill_env_steps': 2000, 'num_prefill_actor_steps': np.float64(17.0), 'num_training_steps_per_epoch': 806}


In [11]:
print("ENV_DIM_ICRL:", {
  "obs_size": env.observation_size,
  "action_size": env.action_size,
  "agents": env.env.agents,
  "num_agents": env.env.num_agents,
  "goal_idx": (args.goal_start_idx, args.goal_end_idx),
})


ENV_DIM_ICRL: {'obs_size': 129, 'action_size': 10, 'agents': ['ally_0', 'ally_1', 'ally_2', 'ally_3', 'ally_4'], 'num_agents': 5, 'goal_idx': (127, 128)}


In [12]:
# print("POST-SMAX ARGS:", {
#         "obs_size": obs_size,
#         "action_size": action_size,
#         "num_agents": args.num_agents,
#         "obs_dim": args.obs_dim,
#         "goal_idx": (args.goal_start_idx, args.goal_end_idx),
#         "discrete_actions": args.discrete_actions,
#     })

In [13]:
#new modification for unavail action penalization
if ('smax' in args.env_id) :
    print("smax and discrete actions")
    get_avail_act = jax.vmap(env.get_avail_actions)

env = envs.training.wrap(
    env,
    episode_length=args.episode_length,
)

obs_size = env.observation_size
action_size = env.action_size
env_keys = jax.random.split(env_key, args.num_envs)
env_state = jax.jit(env.reset)(env_keys)
env.step = jax.jit(env.step)

smax and discrete actions


In [14]:
alph_params, act_params, crtc_params = load_params(args.load_path) if args.load_path else [None]*3


In [15]:
class Actor(nn.Module):
    action_size: int
    norm_type = "layer_norm"

    LOG_STD_MAX = 2
    LOG_STD_MIN = -5

    @nn.compact
    def __call__(self, x):
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)

        mean = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        log_std = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        
        log_std = nn.tanh(log_std)
        log_std = self.LOG_STD_MIN + 0.5 * (self.LOG_STD_MAX - self.LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

# Network setup
# Actor[]
actor = Actor(action_size=action_size)
if not act_params:
    act_params = actor.init(actor_key, np.ones([1, obs_size]))
actor_state = TrainState.create(
    apply_fn=actor.apply,
    params=act_params,
    tx=optax.adam(learning_rate=args.actor_lr)
)


In [16]:

# Critic
sa_encoder = SA_encoder()
sa_encoder_params = sa_encoder.init(sa_key, np.ones([1, args.obs_dim]), np.ones([1, action_size]))
g_encoder = G_encoder()
g_encoder_params = g_encoder.init(g_key, np.ones([1, args.goal_end_idx - args.goal_start_idx]))
c = jnp.asarray(0.0, dtype=jnp.float32)
if not crtc_params:
    crtc_params = {"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params}
critic_state = TrainState.create(
    apply_fn=None,
    params={"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params},
    tx=optax.adam(learning_rate=args.critic_lr),
)

In [17]:
# Entropy coefficient
target_entropy = -0.5 * action_size
log_alpha = jnp.asarray(0.0, dtype=jnp.float32)
if not alph_params:
    alph_params = {"log_alpha": log_alpha}
alpha_state = TrainState.create(
    apply_fn=None,
    params=alph_params,
    tx=optax.adam(learning_rate=args.alpha_lr),
)

In [18]:

# Trainstate
training_state = TrainingState(
    env_steps=jnp.zeros(()),
    gradient_steps=jnp.zeros(()),
    actor_state=actor_state,
    critic_state=critic_state,
    alpha_state=alpha_state,
)

In [19]:

obs_size = env.observation_size
action_size = env.action_size
env_keys = jax.random.split(env_key, args.num_envs)
env_state = jax.jit(env.reset)(env_keys)
env.step = jax.jit(env.step)

#load checkpoint to resume training
alph_params, act_params, crtc_params = load_params(args.load_path) if args.load_path else [None]*3

# Network setup
# Actor
actor = Actor(action_size=action_size)
if not act_params:
    act_params = actor.init(actor_key, np.ones([1, obs_size]))
actor_state = TrainState.create(
    apply_fn=actor.apply,
    params=act_params,
    tx=optax.adam(learning_rate=args.actor_lr)
)

# Critic
sa_encoder = SA_encoder()
sa_encoder_params = sa_encoder.init(sa_key, np.ones([1, args.obs_dim]), np.ones([1, action_size]))
g_encoder = G_encoder()
g_encoder_params = g_encoder.init(g_key, np.ones([1, args.goal_end_idx - args.goal_start_idx]))
c = jnp.asarray(0.0, dtype=jnp.float32)
if not crtc_params:
    crtc_params = {"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params}
critic_state = TrainState.create(
    apply_fn=None,
    params={"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params},
    tx=optax.adam(learning_rate=args.critic_lr),
)

# Entropy coefficient
target_entropy = -0.5 * action_size
log_alpha = jnp.asarray(0.0, dtype=jnp.float32)
if not alph_params:
    alph_params = {"log_alpha": log_alpha}
alpha_state = TrainState.create(
    apply_fn=None,
    params=alph_params,
    tx=optax.adam(learning_rate=args.alpha_lr),
)

# Trainstate
training_state = TrainingState(
    env_steps=jnp.zeros(()),
    gradient_steps=jnp.zeros(()),
    actor_state=actor_state,
    critic_state=critic_state,
    alpha_state=alpha_state,
)

#Replay Buffer
dummy_obs = jnp.zeros((obs_size,))
dummy_action = jnp.zeros((action_size,))

dummy_transition = Transition(
    observation=dummy_obs,
    action=dummy_action,
    reward=0.0,
    discount=0.0,
    extras={
        "state_extras": {
            "truncation": 0.0,
            "seed": 0.0,
        }        
    },
)

def jit_wrap(buffer):
    buffer.insert_internal = jax.jit(buffer.insert_internal)
    buffer.sample_internal = jax.jit(buffer.sample_internal)
    return buffer

#change replay buffer to make the 2nd dimension num_envs_agents
replay_buffer = jit_wrap(
        TrajectoryUniformSamplingQueue(
            max_replay_size=args.max_replay_size,
            dummy_data_sample=dummy_transition,
            sample_batch_size=args.batch_size,
            num_envs=args.num_envs_agents,
            episode_length=args.episode_length,
        )
    )
buffer_state = jax.jit(replay_buffer.init)(buffer_key)

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def deterministic_actor_step(training_state, env, env_state, extra_fields):
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, _ = actor.apply(training_state.actor_state.params, obs)
    action = None
    
    if not args.discrete_actions:
        actions = nn.tanh( means )
    else:
        actions = jnp.argmax(means, axis=-1)
    
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
    nstate = env.step(env_state, actions_)
    
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}
    return nstate, Transition(
        observation=obs,
        action=actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def actor_step(actor_state, env, env_state, key, extra_fields):
    #perform inference to get actions
    keys = jax.random.split(key, args.num_agents)
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, log_stds = actor.apply(actor_state.params, obs)
    actions = None
    transition_actions = None
    
    if not args.discrete_actions:
        stds = jnp.exp(log_stds)
        noise = jax.vmap(lambda rng: jax.random.normal(
                    rng, shape=(args.num_envs,) + means.shape[1:], 
                    dtype=means.dtype), in_axes=0, out_axes=1)(keys)
        noise_ = jnp.reshape(noise, (-1,) + noise.shape[2:])
        actions = nn.tanh( means + stds * noise_)
        transition_actions = actions
    else:
        avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
        avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
        action_logits = means - ((1-avail_actions)*1e10)
        
        pi = distrax.Categorical(logits = action_logits)
        actions = pi.sample(seed = keys[0])
        transition_actions = jax.nn.one_hot(actions, means.shape[1])
        
    #step our environment
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
#        jax.debug.print("actions from first env: {}", transition_actions[0])
    nstate = env.step(env_state, actions_)
    
    #generate an array of transitions, with shape (num_envs*num_agents,...)
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}

    return nstate, Transition(
        observation=obs,
        action=transition_actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

@jax.jit
def get_experience(actor_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused_t):
        env_state, current_key = carry
        current_key, next_key = jax.random.split(current_key)
        env_state, transition = actor_step(actor_state, env, env_state, current_key, extra_fields=("truncation", "seed"))
        return (env_state, next_key), transition

    (env_state, _), data = jax.lax.scan(f, (env_state, key), (), length=args.unroll_length)
    buffer_state = replay_buffer.insert(buffer_state, data)
    return env_state, buffer_state

def prefill_replay_buffer(training_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused):
        del unused
        training_state, env_state, buffer_state, key = carry
        key, new_key = jax.random.split(key)
        env_state, buffer_state = get_experience(
            training_state.actor_state,
            env_state,
            buffer_state,
            key,
        
        )
        training_state = training_state.replace(
            env_steps=training_state.env_steps + args.env_steps_per_actor_step,
        )
        return (training_state, env_state, buffer_state, new_key), ()

    return jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_prefill_actor_steps)[0]

#modified to accomodate discrete action space
@jax.jit
def update_actor_and_alpha(transitions, training_state, key):
    def actor_loss(actor_params, critic_params, log_alpha, transitions, key):
        obs = transitions.observation           # expected_shape = batch_size, obs_size + goal_size
        state = obs[:, :args.obs_dim]
        future_state = transitions.extras["future_state"]
        goal = future_state[:, args.goal_start_idx : args.goal_end_idx]
        observation = jnp.concatenate([state, goal], axis=1)
        jax.debug.print("observation: {}", observation.shape)
        jax.debug.print("observation: {}", observation[0,-2:])

        means, log_stds = actor.apply(actor_params, observation)
        action = None
        log_prob = None
        
        if not args.discrete_actions:
            stds = jnp.exp(log_stds)
            x_ts = means + stds * jax.random.normal(key, shape=means.shape, dtype=means.dtype)
            action = nn.tanh(x_ts)
            log_prob = jax.scipy.stats.norm.logpdf(x_ts, loc=means, scale=stds)
            log_prob -= jnp.log((1 - jnp.square(action)) + 1e-6)
            log_prob = log_prob.sum(-1)           # dimension = B
        else:
            pi = distrax.Categorical(logits = means)
            action = pi.sample(seed = key)
#                jax.debug.print("actions on sampled obs: {}", means[:6,:])
            log_prob = pi.log_prob(action)
            action = jax.nn.one_hot(action, means.shape[1])

        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        sa_repr = sa_encoder.apply(sa_encoder_params, state, action)
        g_repr = g_encoder.apply(g_encoder_params, goal)

        qf_pi = -jnp.sqrt(jnp.sum((sa_repr - g_repr) ** 2, axis=-1))

        actor_loss = jnp.mean( jnp.exp(log_alpha) * log_prob - (qf_pi) )

        return actor_loss, log_prob

    def alpha_loss(alpha_params, log_prob):
        alpha = jnp.exp(alpha_params["log_alpha"])
        alpha_loss = alpha * jnp.mean(jax.lax.stop_gradient(-log_prob - target_entropy))
        return jnp.mean(alpha_loss)
    
    (actorloss, log_prob), actor_grad = jax.value_and_grad(actor_loss, has_aux=True)(training_state.actor_state.params, training_state.critic_state.params, training_state.alpha_state.params['log_alpha'], transitions, key)
    new_actor_state = training_state.actor_state.apply_gradients(grads=actor_grad)

    alphaloss, alpha_grad = jax.value_and_grad(alpha_loss)(training_state.alpha_state.params, log_prob)
    new_alpha_state = training_state.alpha_state.apply_gradients(grads=alpha_grad)

    training_state = training_state.replace(actor_state=new_actor_state, alpha_state=new_alpha_state)

    metrics = {
        "sample_entropy": -log_prob,
        "actor_loss": actorloss,
        "alph_aloss": alphaloss,   
        "log_alpha": training_state.alpha_state.params["log_alpha"],
    }

    return training_state, metrics

@jax.jit
def update_critic(transitions, training_state, key):
    def critic_loss(critic_params, transitions, key):
        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        
        obs = transitions.observation[:, :args.obs_dim]
        action = transitions.action
        
        sa_repr = sa_encoder.apply(sa_encoder_params, obs, action)
        g_repr = g_encoder.apply(g_encoder_params, transitions.observation[:, args.obs_dim:])
        
        # InfoNCE
        logits = -jnp.sqrt(jnp.sum((sa_repr[:, None, :] - g_repr[None, :, :]) ** 2, axis=-1))       # shape = BxB
        critic_loss = -jnp.mean(jnp.diag(logits) - jax.nn.logsumexp(logits, axis=1))

        # logsumexp regularisation
        logsumexp = jax.nn.logsumexp(logits + 1e-6, axis=1)
        critic_loss += args.logsumexp_penalty_coeff * jnp.mean(logsumexp**2)

        I = jnp.eye(logits.shape[0])
        correct = jnp.argmax(logits, axis=1) == jnp.argmax(I, axis=1)
        logits_pos = jnp.sum(logits * I) / jnp.sum(I)
        logits_neg = jnp.sum(logits * (1 - I)) / jnp.sum(1 - I)

        return critic_loss, (logsumexp, I, correct, logits_pos, logits_neg)
        
    (loss, (logsumexp, I, correct, logits_pos, logits_neg)), grad = jax.value_and_grad(critic_loss, has_aux=True)(training_state.critic_state.params, transitions, key)
    new_critic_state = training_state.critic_state.apply_gradients(grads=grad)
    training_state = training_state.replace(critic_state = new_critic_state)

    metrics = {
        "categorical_accuracy": jnp.mean(correct),
        "logits_pos": logits_pos,
        "logits_neg": logits_neg,
        "logsumexp": logsumexp.mean(),
        "critic_loss": loss,
    }

    return training_state, metrics

@jax.jit
def sgd_step(carry, transitions):
    training_state, key = carry
    key, critic_key, actor_key, = jax.random.split(key, 3)

    training_state, actor_metrics = update_actor_and_alpha(transitions, training_state, actor_key)

    training_state, critic_metrics = update_critic(transitions, training_state, critic_key)

    training_state = training_state.replace(gradient_steps = training_state.gradient_steps + 1)

    metrics = {}
    metrics.update(actor_metrics)
    metrics.update(critic_metrics)
    
    return (training_state, key,), metrics

@jax.jit
def training_step(training_state, env_state, buffer_state, key):
    experience_key1, experience_key2, sampling_key, training_key = jax.random.split(key, 4)

    # update buffer
    env_state, buffer_state = get_experience(
        training_state.actor_state,
        env_state,
        buffer_state,
        experience_key1,
    )

    training_state = training_state.replace(
        env_steps=training_state.env_steps + args.env_steps_per_actor_step,
    )

    # sample actor-step worth of transitions
    buffer_state, transitions = replay_buffer.sample(buffer_state)

    # process transitions for training
    batch_keys = jax.random.split(sampling_key, transitions.observation.shape[0])
    transitions = jax.vmap(TrajectoryUniformSamplingQueue.flatten_crl_fn, in_axes=(None, 0, 0))(
        (args.gamma, args.obs_dim, args.goal_start_idx, args.goal_end_idx), transitions, batch_keys
    )
    
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1,) + x.shape[2:], order="F"),
        transitions,
    )
    permutation = jax.random.permutation(experience_key2, len(transitions.observation))
    transitions = jax.tree_util.tree_map(lambda x: x[permutation], transitions)
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1, args.batch_size) + x.shape[1:]),
        transitions,
    )

    # take actor-step worth of training-step
    (training_state, _,), metrics = jax.lax.scan(sgd_step, (training_state, training_key), transitions)

    return (training_state, env_state, buffer_state,), metrics

@jax.jit
def training_epoch(
    training_state,
    env_state,
    buffer_state,
    key,
):  
    @jax.jit
    def f(carry, unused_t):
        ts, es, bs, k = carry
        k, train_key = jax.random.split(k, 2)
        (ts, es, bs,), metrics = training_step(ts, es, bs, train_key)
        return (ts, es, bs, k), metrics

    (training_state, env_state, buffer_state, key), metrics = jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_training_steps_per_epoch)
    
    metrics["buffer_current_size"] = replay_buffer.size(buffer_state)
    return training_state, env_state, buffer_state, metrics

key, prefill_key = jax.random.split(key, 2)

training_state, env_state, buffer_state, _ = prefill_replay_buffer(
    training_state, env_state, buffer_state, prefill_key
)

'''Setting up evaluator'''
evaluator = CrlEvaluator(
    deterministic_actor_step,
    env,
    num_eval_envs=args.num_eval_envs,
    episode_length=args.episode_length,
    key=eval_env_key,
)

In [22]:
training_walltime = 0
print('starting training....')
for ne in range(args.num_epochs):
    t = time.time()

    key, epoch_key = jax.random.split(key)
    training_state, env_state, buffer_state, metrics = training_epoch(training_state, env_state, buffer_state, epoch_key)
    
    metrics = jax.tree_util.tree_map(jnp.mean, metrics)
    metrics = jax.tree_util.tree_map(lambda x: x.block_until_ready(), metrics)

    epoch_training_time = time.time() - t
    training_walltime += epoch_training_time

    sps = (args.env_steps_per_actor_step * args.num_training_steps_per_epoch) / epoch_training_time
    metrics = {
        "training/sps": sps,
        "training/walltime": training_walltime,
        "training/envsteps": training_state.env_steps.item(),
        **{f"training/{name}": value for name, value in metrics.items()},
    }

    # update metrics, get EVAL episode returns
    metrics = evaluator.run_evaluation(training_state, metrics)


starting training....


TypeError: cannot reshape array of shape (1000, 129) (size 129000) into shape (-1, 256, 129) because the product of specified axis sizes (33024) does not evenly divide 129000

In [21]:

#Replay Buffer
dummy_obs = jnp.zeros((obs_size,))
dummy_action = jnp.zeros((action_size,))

dummy_transition = Transition(
    observation=dummy_obs,
    action=dummy_action,
    reward=0.0,
    discount=0.0,
    extras={
        "state_extras": {
            "truncation": 0.0,
            "seed": 0.0,
        }        
    },
)

def jit_wrap(buffer):
    buffer.insert_internal = jax.jit(buffer.insert_internal)
    buffer.sample_internal = jax.jit(buffer.sample_internal)
    return buffer

#change replay buffer to make the 2nd dimension num_envs_agents
replay_buffer = jit_wrap(
        TrajectoryUniformSamplingQueue(
            max_replay_size=args.max_replay_size,
            dummy_data_sample=dummy_transition,
            sample_batch_size=args.batch_size,
            num_envs=args.num_envs_agents,
            episode_length=args.episode_length,
        )
    )
buffer_state = jax.jit(replay_buffer.init)(buffer_key)


In [103]:
replay_buffer._data_shape

(5000, 10, 143)

ValueError: split accepts a single key, but was given a key array of shape (2, 2) != (). Use jax.vmap for batching.

## training

In [104]:

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def deterministic_actor_step(training_state, env, env_state, extra_fields):
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, _ = actor.apply(training_state.actor_state.params, obs)
    action = None
    
    if not args.discrete_actions:
        actions = nn.tanh( means )
    else:
        actions = jnp.argmax(means, axis=-1)
    
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
    nstate = env.step(env_state, actions_)
    
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}
    return nstate, Transition(
        observation=obs,
        action=actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def actor_step(actor_state, env, env_state, key, extra_fields):
    #perform inference to get actions
    keys = jax.random.split(key, args.num_agents)
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    # jax.debug.print("obs: {}", obs[0,-2:])
    # jax.debug.print("goal: {}", obs[:, args.goal_start_idx : args.goal_end_idx][0])
    means, log_stds = actor.apply(actor_state.params, obs)
    actions = None
    transition_actions = None
    
    if not args.discrete_actions:
        stds = jnp.exp(log_stds)
        noise = jax.vmap(lambda rng: jax.random.normal(
                    rng, shape=(args.num_envs,) + means.shape[1:], 
                    dtype=means.dtype), in_axes=0, out_axes=1)(keys)
        noise_ = jnp.reshape(noise, (-1,) + noise.shape[2:])
        actions = nn.tanh( means + stds * noise_)
        transition_actions = actions
    else:
        avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
        avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
        action_logits = means - ((1-avail_actions)*1e10)
        
        pi = distrax.Categorical(logits = action_logits)
        actions = pi.sample(seed = keys[0])
        transition_actions = jax.nn.one_hot(actions, means.shape[1])
        
    #step our environment
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
#        jax.debug.print("actions from first env: {}", transition_actions[0])
    nstate = env.step(env_state, actions_)
    
    #generate an array of transitions, with shape (num_envs*num_agents,...)
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}

    return nstate, Transition(
        observation=obs,
        action=transition_actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

@jax.jit
def get_experience(actor_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused_t):
        env_state, current_key = carry
        current_key, next_key = jax.random.split(current_key)
        env_state, transition = actor_step(actor_state, env, env_state, current_key, extra_fields=("truncation", "seed"))
        return (env_state, next_key), transition

    (env_state, _), data = jax.lax.scan(f, (env_state, key), (), length=args.unroll_length)
    buffer_state = replay_buffer.insert(buffer_state, data)
    return env_state, buffer_state

def prefill_replay_buffer(training_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused):
        del unused
        training_state, env_state, buffer_state, key = carry
        key, new_key = jax.random.split(key)
        env_state, buffer_state = get_experience(
            training_state.actor_state,
            env_state,
            buffer_state,
            key,
        
        )
        training_state = training_state.replace(
            env_steps=training_state.env_steps + args.env_steps_per_actor_step,
        )
        return (training_state, env_state, buffer_state, new_key), ()

    return jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_prefill_actor_steps)[0]

#modified to accomodate discrete action space
@jax.jit
def update_actor_and_alpha(transitions, training_state, key):
    def actor_loss(actor_params, critic_params, log_alpha, transitions, key):
        obs = transitions.observation           # expected_shape = batch_size, obs_size + goal_size
        state = obs[:, :args.obs_dim]
        future_state = transitions.extras["future_state"]
        goal = future_state[:, args.goal_start_idx : args.goal_end_idx]
        
        observation = jnp.concatenate([state, goal], axis=1)
        jax.debug.print("observation: {}", observation.shape)
        jax.debug.print("observation: {}", observation[0,-2:])

        means, log_stds = actor.apply(actor_params, observation)
        action = None
        log_prob = None
        
        if not args.discrete_actions:
            stds = jnp.exp(log_stds)
            x_ts = means + stds * jax.random.normal(key, shape=means.shape, dtype=means.dtype)
            action = nn.tanh(x_ts)
            log_prob = jax.scipy.stats.norm.logpdf(x_ts, loc=means, scale=stds)
            log_prob -= jnp.log((1 - jnp.square(action)) + 1e-6)
            log_prob = log_prob.sum(-1)           # dimension = B
        else:
            pi = distrax.Categorical(logits = means)
            action = pi.sample(seed = key)
#                jax.debug.print("actions on sampled obs: {}", means[:6,:])
            log_prob = pi.log_prob(action)
            action = jax.nn.one_hot(action, means.shape[1])

        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        sa_repr = sa_encoder.apply(sa_encoder_params, state, action)
        g_repr = g_encoder.apply(g_encoder_params, goal)

        qf_pi = -jnp.sqrt(jnp.sum((sa_repr - g_repr) ** 2, axis=-1))

        actor_loss = jnp.mean( jnp.exp(log_alpha) * log_prob - (qf_pi) )

        return actor_loss, log_prob

    def alpha_loss(alpha_params, log_prob):
        alpha = jnp.exp(alpha_params["log_alpha"])
        alpha_loss = alpha * jnp.mean(jax.lax.stop_gradient(-log_prob - target_entropy))
        return jnp.mean(alpha_loss)
    
    (actorloss, log_prob), actor_grad = jax.value_and_grad(actor_loss, has_aux=True)(training_state.actor_state.params, training_state.critic_state.params, training_state.alpha_state.params['log_alpha'], transitions, key)
    new_actor_state = training_state.actor_state.apply_gradients(grads=actor_grad)

    alphaloss, alpha_grad = jax.value_and_grad(alpha_loss)(training_state.alpha_state.params, log_prob)
    new_alpha_state = training_state.alpha_state.apply_gradients(grads=alpha_grad)

    training_state = training_state.replace(actor_state=new_actor_state, alpha_state=new_alpha_state)

    metrics = {
        "sample_entropy": -log_prob,
        "actor_loss": actorloss,
        "alph_aloss": alphaloss,   
        "log_alpha": training_state.alpha_state.params["log_alpha"],
    }

    return training_state, metrics

@jax.jit
def update_critic(transitions, training_state, key):
    def critic_loss(critic_params, transitions, key):
        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        
        obs = transitions.observation[:, :args.obs_dim]
        action = transitions.action
        
        sa_repr = sa_encoder.apply(sa_encoder_params, obs, action)
        g_repr = g_encoder.apply(g_encoder_params, transitions.observation[:, args.obs_dim:])
        
        # InfoNCE
        logits = -jnp.sqrt(jnp.sum((sa_repr[:, None, :] - g_repr[None, :, :]) ** 2, axis=-1))       # shape = BxB
        critic_loss = -jnp.mean(jnp.diag(logits) - jax.nn.logsumexp(logits, axis=1))

        # logsumexp regularisation
        logsumexp = jax.nn.logsumexp(logits + 1e-6, axis=1)
        critic_loss += args.logsumexp_penalty_coeff * jnp.mean(logsumexp**2)

        I = jnp.eye(logits.shape[0])
        correct = jnp.argmax(logits, axis=1) == jnp.argmax(I, axis=1)
        logits_pos = jnp.sum(logits * I) / jnp.sum(I)
        logits_neg = jnp.sum(logits * (1 - I)) / jnp.sum(1 - I)

        return critic_loss, (logsumexp, I, correct, logits_pos, logits_neg)
        
    (loss, (logsumexp, I, correct, logits_pos, logits_neg)), grad = jax.value_and_grad(critic_loss, has_aux=True)(training_state.critic_state.params, transitions, key)
    new_critic_state = training_state.critic_state.apply_gradients(grads=grad)
    training_state = training_state.replace(critic_state = new_critic_state)

    metrics = {
        "categorical_accuracy": jnp.mean(correct),
        "logits_pos": logits_pos,
        "logits_neg": logits_neg,
        "logsumexp": logsumexp.mean(),
        "critic_loss": loss,
    }

    return training_state, metrics

@jax.jit
def sgd_step(carry, transitions):
    training_state, key = carry
    key, critic_key, actor_key, = jax.random.split(key, 3)

    training_state, actor_metrics = update_actor_and_alpha(transitions, training_state, actor_key)

    training_state, critic_metrics = update_critic(transitions, training_state, critic_key)

    training_state = training_state.replace(gradient_steps = training_state.gradient_steps + 1)

    metrics = {}
    metrics.update(actor_metrics)
    metrics.update(critic_metrics)
    
    return (training_state, key,), metrics

@jax.jit
def training_step(training_state, env_state, buffer_state, key):
    experience_key1, experience_key2, sampling_key, training_key = jax.random.split(key, 4)

    # update buffer
    env_state, buffer_state = get_experience(
        training_state.actor_state,
        env_state,
        buffer_state,
        experience_key1,
    )

    training_state = training_state.replace(
        env_steps=training_state.env_steps + args.env_steps_per_actor_step,
    )

    # sample actor-step worth of transitions
    buffer_state, transitions = replay_buffer.sample(buffer_state)

    # process transitions for training
    batch_keys = jax.random.split(sampling_key, transitions.observation.shape[0])
    transitions = jax.vmap(TrajectoryUniformSamplingQueue.flatten_crl_fn, in_axes=(None, 0, 0))(
        (args.gamma, args.obs_dim, args.goal_start_idx, args.goal_end_idx), transitions, batch_keys
    )
    
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1,) + x.shape[2:], order="F"),
        transitions,
    )
    permutation = jax.random.permutation(experience_key2, len(transitions.observation))
    transitions = jax.tree_util.tree_map(lambda x: x[permutation], transitions)
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1, args.batch_size) + x.shape[1:]),
        transitions,
    )

    # take actor-step worth of training-step
    (training_state, _,), metrics = jax.lax.scan(sgd_step, (training_state, training_key), transitions)

    return (training_state, env_state, buffer_state,), metrics

@jax.jit
def training_epoch(
    training_state,
    env_state,
    buffer_state,
    key,
):  
    @jax.jit
    def f(carry, unused_t):
        ts, es, bs, k = carry
        k, train_key = jax.random.split(k, 2)
        (ts, es, bs,), metrics = training_step(ts, es, bs, train_key)
        return (ts, es, bs, k), metrics

    (training_state, env_state, buffer_state, key), metrics = jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_training_steps_per_epoch)
    
    metrics["buffer_current_size"] = replay_buffer.size(buffer_state)
    return training_state, env_state, buffer_state, metrics


In [105]:

key, prefill_key = jax.random.split(key, 2)

training_state, env_state, buffer_state, _ = prefill_replay_buffer(
    training_state, env_state, buffer_state, prefill_key
)

'''Setting up evaluator'''
evaluator = CrlEvaluator(
    deterministic_actor_step,
    env,
    num_eval_envs=args.num_eval_envs,
    episode_length=args.episode_length,
    key=eval_env_key,
)


In [ ]:


training_walltime = 0
print('starting training....')
for ne in range(args.num_epochs):
    t = time.time()

    key, epoch_key = jax.random.split(key)
    training_state, env_state, buffer_state, metrics = training_epoch(training_state, env_state, buffer_state, epoch_key)
    
    metrics = jax.tree_util.tree_map(jnp.mean, metrics)
    metrics = jax.tree_util.tree_map(lambda x: x.block_until_ready(), metrics)

    epoch_training_time = time.time() - t
    training_walltime += epoch_training_time

    sps = (args.env_steps_per_actor_step * args.num_training_steps_per_epoch) / epoch_training_time
    metrics = {
        "training/sps": sps,
        "training/walltime": training_walltime,
        "training/envsteps": training_state.env_steps.item(),
        **{f"training/{name}": value for name, value in metrics.items()},
    }

    # update metrics, get EVAL episode returns
    metrics = evaluator.run_evaluation(training_state, metrics)
    print(metrics)


    if args.checkpoint:
        # Save current policy and critic params.
        params = (training_state.alpha_state.params, training_state.actor_state.params, training_state.critic_state.params)
        path = f"{save_path}/step_{int(training_state.env_steps)}.pkl"
        save_params(path, params)
    
    if args.track:
        wandb.log(metrics, step=ne*args.num_training_steps_per_epoch*args.env_steps_per_actor_step) #modified to have x-axis of plots be training_steps instead of epochs

        if args.wandb_mode == 'offline':
            trigger_sync()

if args.checkpoint:
    # Save current policy and critic params.
    params = (training_state.alpha_state.params, training_state.actor_state.params, training_state.critic_state.params)
    path = f"{save_path}/final.pkl"
    save_params(path, params)

    
# (50000000 - 1024 x 1000) / 50 x 1024 x 62 = 15        #number of actor steps per epoch (which is equal to the number of training steps)
# 1024 x 999 / 256 = 4000                               #number of gradient steps per actor step 
# 1024 x 62 / 4000 = 16                                 #ratio of env steps per gradient step




starting training....


TypeError: cannot reshape array of shape (1000, 129) (size 129000) into shape (-1, 256, 129) because the product of specified axis sizes (33024) does not evenly divide 129000

https://symbolize.stripped_domain/r/?trace=7ff189525e9e,7ff18944251f&map= 
*** SIGTERM received by PID 2322685 (TID 2322685) on cpu 166 from PID 2338776; stack trace: ***
PC: @     0x7ff189525e9e  (unknown)  epoll_wait
    @     0x7feff2db47e5       1904  (unknown)
    @     0x7ff189442520  (unknown)  (unknown)
https://symbolize.stripped_domain/r/?trace=7ff189525e9e,7feff2db47e4,7ff18944251f&map= 
E1118 15:31:30.441139 2322685 coredump_hook.cc:247] RAW: Remote crash gathering disabled for SIGTERM.


: 

# new

## setup

In [40]:
import os
import jax
import flax
import tyro
import time
import optax
import wandb
import pickle
import random
import wandb_osh
import numpy as np
import flax.linen as nn
import jax.numpy as jnp
import distrax

from brax import envs
from etils import epath
from dataclasses import dataclass
from collections import namedtuple
from typing import NamedTuple, Any
from wandb_osh.hooks import TriggerWandbSyncHook
from flax.training.train_state import TrainState
from flax.linen.initializers import variance_scaling

from evaluator import CrlEvaluator
from buffer import TrajectoryUniformSamplingQueue

import matplotlib.pyplot as plt

#with the multi-policy idea, all that would be need to be changed would be many different actor_params in the actor_state

#list of modifications so far:
#actor_step, deterministic_actor_step, simple env if statement, replay buffer initialization, checkpoint loading

#intended modifications:
#decrease the max replay size in accordance with num agents
#notice that this gives the same effect as not making new buffers for agents and
#instead putting the agent trajectories one after the other in the same buffer

@dataclass
class Args:
    exp_name: str = "testing"
    seed: int = 1
    torch_deterministic: bool = True
    cuda: bool = True
    track: bool = True
    wandb_project_name: str = "ICRL_Reproduction"
    wandb_entity: str = 'asim_awad'
    wandb_mode: str = 'offline'
    wandb_dir: str = '.'
    wandb_group: str = '.'
    capture_video: bool = False
    checkpoint: bool = False
    load_path: str = ''

    #environment specific arguments
    env_id: str = "smax"
    smax_map_name: str = "2s3z"
    episode_length: int = 101
    # to be filled in runtime
    obs_dim: int = 0
    goal_start_idx: int = 0
    goal_end_idx: int = 0
    num_agents: int = 0
    discrete_actions: bool = False

    # Algorithm specific arguments
    total_env_steps: int = 50_000_000
    num_epochs: int = 500
    num_envs: int = 256
    num_eval_envs: int = 64
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    batch_size: int = 256
    gamma: float = 0.99
    logsumexp_penalty_coeff: float = 0.1

    max_replay_size: int = 5000 #temporarily reduced by 2 to accomodate more agents, should make this systematic in future
    min_replay_size: int = 1000
    
    unroll_length: int  = 62

    # to be filled in runtime
    env_steps_per_actor_step : int = 0
    """number of env steps per actor step (computed in runtime)"""
    num_prefill_env_steps : int = 0
    """number of env steps to fill the buffer before starting training (computed in runtime)"""
    num_prefill_actor_steps : int = 0
    """number of actor steps to fill the buffer before starting training (computed in runtime)"""
    num_training_steps_per_epoch : int = 0
    """the number of training steps per epoch(computed in runtime)"""
    num_envs_agents : int = 0
    """the number of agents times the number of environments (2nd dimension replay buffer)"""

class SA_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, s: jnp.ndarray, a: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros
        
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = jnp.concatenate([s, a], axis=-1)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x
    
class G_encoder(nn.Module):
    norm_type = "layer_norm"
    @nn.compact
    def __call__(self, g: jnp.ndarray):

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(g)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(64, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        return x

class Actor(nn.Module):
    action_size: int
    norm_type = "layer_norm"

    LOG_STD_MAX = 5
    LOG_STD_MIN = -5

    @nn.compact
    def __call__(self, x):
        if self.norm_type == "layer_norm":
            normalize = lambda x: nn.LayerNorm()(x)
        else:
            normalize = lambda x: x

        lecun_unfirom = variance_scaling(1/3, "fan_in", "uniform")
        bias_init = nn.initializers.zeros

        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)
        x = nn.Dense(1024, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        x = normalize(x)
        x = nn.swish(x)

        mean = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        log_std = nn.Dense(self.action_size, kernel_init=lecun_unfirom, bias_init=bias_init)(x)
        
        log_std = nn.tanh(log_std)
        log_std = self.LOG_STD_MIN + 0.5 * (self.LOG_STD_MAX - self.LOG_STD_MIN) * (log_std + 1)  # From SpinUp / Denis Yarats

        return mean, log_std

@flax.struct.dataclass
class TrainingState:
    """Contains training state for the learner"""
    env_steps: jnp.ndarray
    gradient_steps: jnp.ndarray
    actor_state: TrainState
    critic_state: TrainState
    alpha_state: TrainState

class Transition(NamedTuple):
    """Container for a transition"""
    observation: jnp.ndarray
    action: jnp.ndarray
    reward: jnp.ndarray
    discount: jnp.ndarray
    avail_actions: jnp.ndarray
    extras: jnp.ndarray = ()  

def load_params(path: str):
    with epath.Path(path).open('rb') as fin:
        buf = fin.read()
    return pickle.loads(buf)

def save_params(path: str, params: Any):
    """Saves parameters in flax format."""
    with epath.Path(path).open('wb') as fout:
        fout.write(pickle.dumps(params))
         

## code

In [42]:

args = Args()

args.env_steps_per_actor_step = args.num_envs * args.unroll_length
args.num_prefill_env_steps = args.min_replay_size * args.num_envs
args.num_prefill_actor_steps = np.ceil(args.min_replay_size / args.unroll_length)
args.num_training_steps_per_epoch = (args.total_env_steps - args.num_prefill_env_steps) // (args.num_epochs * args.env_steps_per_actor_step)

run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

if args.track:

    if args.wandb_group ==  '.':
        args.wandb_group = None
        
    wandb.init(
        project=args.wandb_project_name,
        entity=args.wandb_entity,
        mode=args.wandb_mode,
        group=args.wandb_group,
        dir=args.wandb_dir,
        config=vars(args),
        name=run_name,
        monitor_gym=True,
        save_code=True,
    )

    if args.wandb_mode == 'offline':
        wandb_osh.set_log_level("ERROR")
        trigger_sync = TriggerWandbSyncHook()
    
if args.checkpoint:
    from pathlib import Path
    save_path = Path(args.wandb_dir) / Path(run_name)
    os.mkdir(path=save_path)

random.seed(args.seed)
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, buffer_key, env_key, eval_env_key, actor_key, sa_key, g_key = jax.random.split(key, 7)

# Environment setup    
if args.env_id == "ant":
    from envs.ant import Ant
    env = Ant(
        backend="spring",
        exclude_current_positions_from_observation=False,
        terminate_when_unhealthy=True,
    )

    args.obs_dim = 29
    args.goal_start_idx = 0
    args.goal_end_idx = 2

elif "maze" in args.env_id:
    from envs.ant_maze import AntMaze
    env = AntMaze(
        backend="spring",
        exclude_current_positions_from_observation=False,
        terminate_when_unhealthy=True,
        maze_layout_name=args.env_id[4:]
    )

    args.obs_dim = 29
    args.goal_start_idx = 0
    args.goal_end_idx = 2

elif args.env_id == "simple_marl":
    from envs.mpe_simple_marl import SimpleMPEMARL
    env = SimpleMPEMARL()
    args.obs_dim = 4
    args.goal_start_idx = 2
    args.goal_end_idx = 4
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mpe_tag":
    from envs.mpe_tag import MPETagCoop
    env = MPETagCoop()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_adversaries
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mpe_tag_facmac":
    from envs.mpe_tag_facmac import MPETagFacmac
    env = MPETagFacmac()
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1
    args.num_agents = env.env.num_adversaries
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "smax":
    from envs.smax import SmaxEnv
    env = SmaxEnv(map_name=args.smax_map_name)
    # args.obs_dim = env.observation_size - env.env.num_enemies
    # args.goal_start_idx = env.observation_size - env.env.num_enemies*2
    # args.goal_end_idx = env.observation_size - env.env.num_enemies
    
    args.obs_dim = env.observation_size - 1
    args.goal_start_idx = env.observation_size - 2
    args.goal_end_idx = env.observation_size - 1

    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents
    args.discrete_actions = False #for now

elif args.env_id == "smax_move":
    from envs.smax_move import SmaxEnv
    env = SmaxEnv()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents
    args.discrete_actions = False #for now

elif args.env_id == "mabrax_ant":
    from envs.mabrax_ant import MABraxAnt
    env = MABraxAnt()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

elif args.env_id == "mabrax_walker":
    from envs.mabrax_walker import MABraxWalker
    env = MABraxWalker()
    args.obs_dim = env.observation_size - 2
    args.goal_start_idx = env.observation_size - 4
    args.goal_end_idx = env.observation_size - 2
    args.num_agents = env.env.num_agents
    args.num_envs_agents = args.num_envs * args.num_agents

else:
    raise NotImplementedError

#new modification for unavail action penalization
if 'smax' in args.env_id:
    get_avail_act = jax.vmap(env.get_avail_actions)

env = envs.training.wrap(
    env,
    episode_length=args.episode_length,
)

obs_size = env.observation_size
action_size = env.action_size
env_keys = jax.random.split(env_key, args.num_envs)
env_state = jax.jit(env.reset)(env_keys)
env.step = jax.jit(env.step)

#load checkpoint to resume training
alph_params, act_params, crtc_params = load_params(args.load_path) if args.load_path else [None]*3

# Network setup
# Actor
actor = Actor(action_size=action_size)
if not act_params:
    act_params = actor.init(actor_key, np.ones([1, obs_size]))
actor_state = TrainState.create(
    apply_fn=actor.apply,
    params=act_params,
    tx=optax.adam(learning_rate=args.actor_lr)
)

# Critic
sa_encoder = SA_encoder()
sa_encoder_params = sa_encoder.init(sa_key, np.ones([1, args.obs_dim]), np.ones([1, action_size]))
g_encoder = G_encoder()
g_encoder_params = g_encoder.init(g_key, np.ones([1, args.goal_end_idx - args.goal_start_idx]))
c = jnp.asarray(0.0, dtype=jnp.float32)
if not crtc_params:
    crtc_params = {"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params}
critic_state = TrainState.create(
    apply_fn=None,
    params={"sa_encoder": sa_encoder_params, "g_encoder": g_encoder_params},
    tx=optax.adam(learning_rate=args.critic_lr),
)

# Entropy coefficient
target_entropy = -0.5 * action_size
log_alpha = jnp.asarray(0.0, dtype=jnp.float32)
if not alph_params:
    alph_params = {"log_alpha": log_alpha}
alpha_state = TrainState.create(
    apply_fn=None,
    params=alph_params,
    tx=optax.adam(learning_rate=args.alpha_lr),
)

# Trainstate
training_state = TrainingState(
    env_steps=jnp.zeros(()),
    gradient_steps=jnp.zeros(()),
    actor_state=actor_state,
    critic_state=critic_state,
    alpha_state=alpha_state,
)

#Replay Buffer
dummy_obs = jnp.zeros((obs_size,))
dummy_action = jnp.zeros((action_size,))

dummy_transition = Transition(
    observation=dummy_obs,
    action=dummy_action,
    avail_actions=dummy_action,
    reward=0.0,
    discount=0.0,
    extras={
        "state_extras": {
            "truncation": 0.0,
            "seed": 0.0,
        }        
    },
)

def jit_wrap(buffer):
    buffer.insert_internal = jax.jit(buffer.insert_internal)
    buffer.sample_internal = jax.jit(buffer.sample_internal)
    return buffer

#change replay buffer to make the 2nd dimension num_envs_agents
replay_buffer = jit_wrap(
        TrajectoryUniformSamplingQueue(
            max_replay_size=args.max_replay_size,
            dummy_data_sample=dummy_transition,
            sample_batch_size=args.batch_size,
            num_envs=args.num_envs_agents,
            episode_length=args.episode_length,
        )
    )
buffer_state = jax.jit(replay_buffer.init)(buffer_key)


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


## funcitno

In [ ]:

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def deterministic_actor_step(training_state, env, env_state, extra_fields):
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, _ = actor.apply(training_state.actor_state.params, obs)
    action = None
    avail_actions = None
    
    if not args.discrete_actions:
        if 'smax' in args.env_id:
            avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
            avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
            means = means - ((1-avail_actions)*1e10)
            
        actions = nn.tanh( means )
    else:
        actions = jnp.argmax(means, axis=-1)
    
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
    nstate = env.step(env_state, actions_)
    
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}
    return nstate, Transition(
        observation=obs,
        action=actions,
        avail_actions=avail_actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

#modified to have extra dimension for multiple agents
#modified to sample from categorical distribution
def actor_step(actor_state, env, env_state, key, extra_fields):
    #perform inference to get actions
    keys = jax.random.split(key, args.num_agents)
    obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
    means, log_stds = actor.apply(actor_state.params, obs)
    actions = None
    transition_actions = None
    avail_actions = None
    
    if not args.discrete_actions:
        stds = jnp.exp(log_stds)
        noise = jax.vmap(lambda rng: jax.random.gumbel(
                    rng, shape=(args.num_envs,) + means.shape[1:], 
                    dtype=means.dtype), in_axes=0, out_axes=1)(keys) #modified to gumbel distribution for smax
        noise_ = jnp.reshape(noise, (-1,) + noise.shape[2:])
        
        if 'smax' in args.env_id:
            avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
            avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
            means = means - ((1-avail_actions)*1e10)
        
        actions = nn.tanh( means + stds * noise_)

        transition_actions = actions
    else:
        avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
        avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
        action_logits = means - ((1-avail_actions)*1e10)
        
        pi = distrax.Categorical(logits = action_logits)
        actions = pi.sample(seed = keys[0])
        transition_actions = jax.nn.one_hot(actions, means.shape[1])
        
    #step our environment
    actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
    # jax.debug.print("actions from first env: {}", actions_[0][0])
    nstate = env.step(env_state, actions_)
    
    #generate an array of transitions, with shape (num_envs*num_agents,...)
    state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}

    return nstate, Transition(
        observation=obs,
        action=transition_actions,
        avail_actions=avail_actions,
        reward=jnp.repeat(nstate.reward, args.num_agents),
        discount=jnp.repeat(1-nstate.done, args.num_agents),
        extras={"state_extras": state_extras},
    )

@jax.jit
def get_experience(actor_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused_t):
        env_state, current_key = carry
        current_key, next_key = jax.random.split(current_key)
        env_state, transition = actor_step(actor_state, env, env_state, current_key, extra_fields=("truncation", "seed"))
        return (env_state, next_key), transition

    (env_state, _), data = jax.lax.scan(f, (env_state, key), (), length=args.unroll_length)
    buffer_state = replay_buffer.insert(buffer_state, data)
    return env_state, buffer_state

def prefill_replay_buffer(training_state, env_state, buffer_state, key):
    @jax.jit
    def f(carry, unused):
        del unused
        training_state, env_state, buffer_state, key = carry
        key, new_key = jax.random.split(key)
        env_state, buffer_state = get_experience(
            training_state.actor_state,
            env_state,
            buffer_state,
            key,
        
        )
        training_state = training_state.replace(
            env_steps=training_state.env_steps + args.env_steps_per_actor_step,
        )
        return (training_state, env_state, buffer_state, new_key), ()

    return jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_prefill_actor_steps)[0]

#modified to accomodate discrete action space
@jax.jit
def update_actor_and_alpha(transitions, training_state, key):
    def actor_loss(actor_params, critic_params, log_alpha, transitions, key):
        obs = transitions.observation           # expected_shape = batch_size, obs_size + goal_size
        state = obs[:, :args.obs_dim]
        future_state = transitions.extras["future_state"]
        goal = future_state[:, args.goal_start_idx : args.goal_end_idx]
        observation = jnp.concatenate([state, goal], axis=1)
        avail_actions = transitions.avail_actions

        means, log_stds = actor.apply(actor_params, observation)
        action = None
        log_prob = None
        
        if not args.discrete_actions:
            stds = jnp.exp(log_stds)
            means = means - ((1-avail_actions)*1e10)
            x_ts = means + stds * jax.random.gumbel(key, shape=means.shape, dtype=means.dtype) #modified to gumbel noise
            action = nn.tanh(x_ts)
            #log_prob = jax.scipy.stats.norm.logpdf(x_ts, loc=means, scale=stds)
            x_std = (x_ts - means)/stds
            log_prob = -(x_std + jnp.exp(-x_std)) - log_stds #modified to gumbel pdf
            
            log_prob -= jnp.log((1 - jnp.square(action)) + 1e-6)
            log_prob = jnp.where(avail_actions == 0, 0, log_prob) #for unavailable actions prob of action is 1
            log_prob = log_prob.sum(-1)           # dimension = B
        else:
            pi = distrax.Categorical(logits = means)
            action = pi.sample(seed = key)
        #                jax.debug.print("actions on sampled obs: {}", means[:6,:])
            log_prob = pi.log_prob(action)
            action = jax.nn.one_hot(action, means.shape[1])

        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        sa_repr = sa_encoder.apply(sa_encoder_params, state, action)
        g_repr = g_encoder.apply(g_encoder_params, goal)

        qf_pi = -jnp.sqrt(jnp.sum((sa_repr - g_repr) ** 2, axis=-1))

        actor_loss = jnp.mean( jnp.exp(log_alpha) * log_prob - (qf_pi) )

        return actor_loss, log_prob

    def alpha_loss(alpha_params, log_prob):
        alpha = jnp.exp(alpha_params["log_alpha"])
        alpha_loss = alpha * jnp.mean(jax.lax.stop_gradient(-log_prob - target_entropy))
        return jnp.mean(alpha_loss)
    
    (actorloss, log_prob), actor_grad = jax.value_and_grad(actor_loss, has_aux=True)(training_state.actor_state.params, training_state.critic_state.params, training_state.alpha_state.params['log_alpha'], transitions, key)
    new_actor_state = training_state.actor_state.apply_gradients(grads=actor_grad)

    alphaloss, alpha_grad = jax.value_and_grad(alpha_loss)(training_state.alpha_state.params, log_prob)
    new_alpha_state = training_state.alpha_state.apply_gradients(grads=alpha_grad)

    training_state = training_state.replace(actor_state=new_actor_state, alpha_state=new_alpha_state)

    metrics = {
        "sample_entropy": -log_prob,
        "actor_loss": actorloss,
        "alph_aloss": alphaloss,   
        "log_alpha": training_state.alpha_state.params["log_alpha"],
    }

    return training_state, metrics

@jax.jit
def update_critic(transitions, training_state, key):
    def critic_loss(critic_params, transitions, key):
        sa_encoder_params, g_encoder_params = critic_params["sa_encoder"], critic_params["g_encoder"]
        
        obs = transitions.observation[:, :args.obs_dim]
        action = transitions.action
        
        sa_repr = sa_encoder.apply(sa_encoder_params, obs, action)
        g_repr = g_encoder.apply(g_encoder_params, transitions.observation[:, args.obs_dim:])
        
        # InfoNCE
        logits = -jnp.sqrt(jnp.sum((sa_repr[:, None, :] - g_repr[None, :, :]) ** 2, axis=-1))       # shape = BxB
        critic_loss = -jnp.mean(jnp.diag(logits) - jax.nn.logsumexp(logits, axis=1))

        # logsumexp regularisation
        logsumexp = jax.nn.logsumexp(logits + 1e-6, axis=1)
        critic_loss += args.logsumexp_penalty_coeff * jnp.mean(logsumexp**2)

        I = jnp.eye(logits.shape[0])
        correct = jnp.argmax(logits, axis=1) == jnp.argmax(I, axis=1)
        logits_pos = jnp.sum(logits * I) / jnp.sum(I)
        logits_neg = jnp.sum(logits * (1 - I)) / jnp.sum(1 - I)

        return critic_loss, (logsumexp, I, correct, logits_pos, logits_neg)
        
    (loss, (logsumexp, I, correct, logits_pos, logits_neg)), grad = jax.value_and_grad(critic_loss, has_aux=True)(training_state.critic_state.params, transitions, key)
    new_critic_state = training_state.critic_state.apply_gradients(grads=grad)
    training_state = training_state.replace(critic_state = new_critic_state)

    metrics = {
        "categorical_accuracy": jnp.mean(correct),
        "logits_pos": logits_pos,
        "logits_neg": logits_neg,
        "logsumexp": logsumexp.mean(),
        "critic_loss": loss,
    }

    return training_state, metrics

@jax.jit
def sgd_step(carry, transitions):
    training_state, key = carry
    key, critic_key, actor_key, = jax.random.split(key, 3)

    training_state, actor_metrics = update_actor_and_alpha(transitions, training_state, actor_key)

    training_state, critic_metrics = update_critic(transitions, training_state, critic_key)

    training_state = training_state.replace(gradient_steps = training_state.gradient_steps + 1)

    metrics = {}
    metrics.update(actor_metrics)
    metrics.update(critic_metrics)
    
    return (training_state, key,), metrics

@jax.jit
def training_step(training_state, env_state, buffer_state, key):
    experience_key1, experience_key2, sampling_key, training_key = jax.random.split(key, 4)

    # update buffer
    env_state, buffer_state = get_experience(
        training_state.actor_state,
        env_state,
        buffer_state,
        experience_key1,
    )

    training_state = training_state.replace(
        env_steps=training_state.env_steps + args.env_steps_per_actor_step,
    )

    # sample actor-step worth of transitions
    buffer_state, transitions = replay_buffer.sample(buffer_state)

    # process transitions for training
    batch_keys = jax.random.split(sampling_key, transitions.observation.shape[0])
    transitions = jax.vmap(TrajectoryUniformSamplingQueue.flatten_crl_fn, in_axes=(None, 0, 0))(
        (args.gamma, args.obs_dim, args.goal_start_idx, args.goal_end_idx), transitions, batch_keys
    )
    
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1,) + x.shape[2:], order="F"),
        transitions,
    )
    permutation = jax.random.permutation(experience_key2, len(transitions.observation))
    transitions = jax.tree_util.tree_map(lambda x: x[permutation], transitions)
    transitions = jax.tree_util.tree_map(
        lambda x: jnp.reshape(x, (-1, args.batch_size) + x.shape[1:]),
        transitions,
    )

    # take actor-step worth of training-step
    (training_state, _,), metrics = jax.lax.scan(sgd_step, (training_state, training_key), transitions)

    return (training_state, env_state, buffer_state,), metrics

@jax.jit
def training_epoch(
    training_state,
    env_state,
    buffer_state,
    key,
):  
    @jax.jit
    def f(carry, unused_t):
        ts, es, bs, k = carry
        k, train_key = jax.random.split(k, 2)
        (ts, es, bs,), metrics = training_step(ts, es, bs, train_key)
        return (ts, es, bs, k), metrics

    (training_state, env_state, buffer_state, key), metrics = jax.lax.scan(f, (training_state, env_state, buffer_state, key), (), length=args.num_training_steps_per_epoch)
    
    metrics["buffer_current_size"] = replay_buffer.size(buffer_state)
    return training_state, env_state, buffer_state, metrics

key, prefill_key = jax.random.split(key, 2)

training_state, env_state, buffer_state, _ = prefill_replay_buffer(
    training_state, env_state, buffer_state, prefill_key
)


In [44]:
#perform inference to get actions
keys = jax.random.split(key, args.num_agents)
obs = jnp.reshape(env_state.obs, (-1,) + env_state.obs.shape[2:])
means, log_stds = actor.apply(actor_state.params, obs)
actions = None
transition_actions = None
avail_actions = None


In [65]:
env_state, buffer_state = get_experience(
            training_state.actor_state,
            env_state,
            buffer_state,
            key,
        
        )

In [69]:
buffer_state, transitions = replay_buffer.sample(buffer_state)


In [70]:
transitions

Transition(observation=Array([[[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6720834 ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6720834 ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6720834 ,  0.        ],
        ...,
        [ 0.86875004,  0.37295818,  0.10607741, ...,  0.        ,
          4.7454166 ,  0.        ],
        [ 0.86875004,  0.37295818,  0.1605106 , ...,  0.        ,
          4.7454166 ,  0.        ],
        [ 0.86875004, -0.03140015,  0.1605106 , ...,  0.        ,
          4.7454166 ,  0.        ]],

       [[ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6975    ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6975    ,  0.        ],
        [ 0.        ,  0.        ,  0.        , ...,  0.        ,
          4.6975    ,  0.        ],
        ...,
        [ 1.        , -0.123891

In [ ]:
if not args.discrete_actions:
    print("X")
    stds = jnp.exp(log_stds)
    noise = jax.vmap(lambda rng: jax.random.gumbel(
                rng, shape=(args.num_envs,) + means.shape[1:], 
                dtype=means.dtype), in_axes=0, out_axes=1)(keys) #modified to gumbel distribution for smax
    noise_ = jnp.reshape(noise, (-1,) + noise.shape[2:])
    
    if 'smax' in args.env_id:
        avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
        avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
        means = means - ((1-avail_actions)*1e10)
    
    actions = nn.tanh( means + stds * noise_)

    transition_actions = actions

X


In [53]:
actions[0]

Array([-0.3032852 ,  0.9983067 , -0.33133793, -0.9999947 , -0.49648088,
       -0.43504158,  0.99998397,  0.11511769, -1.        , -1.        ],      dtype=float32)

In [68]:
transitions

NameError: name 'transitions' is not defined

In [57]:


# else:
#     avail_actions = get_avail_act(env_state) # modified code to penalize unavailable actions
#     avail_actions = jnp.reshape(avail_actions, (-1,)+avail_actions.shape[2:])
#     action_logits = means - ((1-avail_actions)*1e10)
    
#     pi = distrax.Categorical(logits = action_logits)
#     actions = pi.sample(seed = keys[0])
#     transition_actions = jax.nn.one_hot(actions, means.shape[1])
    
#step our environment
actions_ = jnp.reshape(actions, (-1, args.num_agents,) + actions.shape[1:])
# jax.debug.print("actions from first env: {}", actions_[0][0])
nstate = env.step(env_state, actions_)

#generate an array of transitions, with shape (num_envs*num_agents,...)
# state_extras = {x: jnp.repeat(nstate.info[x], args.num_agents) for x in extra_fields}

# x =  nstate, Transition(
#     observation=obs,
#     action=transition_actions,
#     avail_actions=avail_actions,
#     reward=jnp.repeat(nstate.reward, args.num_agents),
#     discount=jnp.repeat(1-nstate.done, args.num_agents),
#     extras={"state_extras": state_extras},
# )


In [64]:
transition_actions.shape

(1280, 10)

In [62]:
actions_[0][0]

Array([-0.3032852 ,  0.9983067 , -0.33133793, -0.9999947 , -0.49648088,
       -0.43504158,  0.99998397,  0.11511769, -1.        , -1.        ],      dtype=float32)

In [ ]:

'''Setting up evaluator'''
evaluator = CrlEvaluator(
    deterministic_actor_step,
    env,
    num_eval_envs=args.num_eval_envs,
    episode_length=args.episode_length,
    key=eval_env_key,
)

training_walltime = 0
print('starting training....')
for ne in range(args.num_epochs):
    t = time.time()

    key, epoch_key = jax.random.split(key)
    training_state, env_state, buffer_state, metrics = training_epoch(training_state, env_state, buffer_state, epoch_key)
    
    metrics = jax.tree_util.tree_map(jnp.mean, metrics)
    metrics = jax.tree_util.tree_map(lambda x: x.block_until_ready(), metrics)

    epoch_training_time = time.time() - t
    training_walltime += epoch_training_time

    sps = (args.env_steps_per_actor_step * args.num_training_steps_per_epoch) / epoch_training_time
    metrics = {
        "training/sps": sps,
        "training/walltime": training_walltime,
        "training/envsteps": training_state.env_steps.item(),
        **{f"training/{name}": value for name, value in metrics.items()},
    }

    # update metrics, get EVAL episode returns
    metrics = evaluator.run_evaluation(training_state, metrics)
    print(metrics)

    # update TRAINING episode returns
    # train_returns = env_state.info["returned_episode_returns"]
    # mean_train_returns = train_returns[:, :args.num_agents].mean(axis=(0, 1))
    # metrics["training/mean_returned_episode_returns"] = mean_train_returns
    # print("Current Training Episode Return: ", mean_train_returns)

    # make TRAINING reward plot
    # plt.clf()
    # plt.plot(training_episode_returns)
    # print("Current Training Episode Return: ", training_episode_returns[-1])
    # plt.xlabel("Updates")
    # plt.ylabel("Returns")
    # plt.title("ICRL-FF=MPE_TAG_FACMAC")
    # plt.savefig(f"TRAIN_icrl_ff_MPE_TAG_FACMAC_epoch{ne}.png")
    # plt.close()

    # make EVAL reward plot
    # plt.clf()
    # plt.plot(eval_episode_returns)
    # print("Current Evaluation Episode Return: ", eval_episode_returns[-1])
    # plt.xlabel("Updates")
    # plt.ylabel("Returns")
    # plt.title("ICRL-FF=MPE_TAG_FACMAC")
    # plt.savefig(f"EVAL_icrl_ff_MPE_TAG_FACMAC_epoch{ne}.png")
    # plt.close()

    if args.checkpoint:
        # Save current policy and critic params.
        params = (training_state.alpha_state.params, training_state.actor_state.params, training_state.critic_state.params)
        path = f"{save_path}/step_{int(training_state.env_steps)}.pkl"
        save_params(path, params)
    
    if args.track:
        wandb.log(metrics, step=ne*args.num_training_steps_per_epoch*args.env_steps_per_actor_step) #modified to have x-axis of plots be training_steps instead of epochs

        if args.wandb_mode == 'offline':
            trigger_sync()

if args.checkpoint:
    # Save current policy and critic params.
    params = (training_state.alpha_state.params, training_state.actor_state.params, training_state.critic_state.params)
    path = f"{save_path}/final.pkl"
    save_params(path, params)

    
# (50000000 - 1024 x 1000) / 50 x 1024 x 62 = 15        #number of actor steps per epoch (which is equal to the number of training steps)
# 1024 x 999 / 256 = 4000                               #number of gradient steps per actor step 
# 1024 x 62 / 4000 = 16                                 #ratio of env steps per gradient step
